# LAB | Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act

## Learning Objectives
- Build a RAG pipeline over legal documents and a podcast transcript
- Use metadata-aware chunking to support targeted retrieval
- Compare baseline vector retrieval with reranking methods
- Evaluate retrieval quality with Precision@5, MRR, and NDCG@5
- Interpret when reranking is useful and when a strong baseline is enough

## Pipeline Architecture
```text
DATA -> CHUNKING -> EMBEDDING -> VECTOR DB -> QUERY EMBEDDING -> RETRIEVAL -> RERANKING -> GENERATION
```

The lab uses a small, domain-focused corpus: EU AI Act material, trustworthy AI guidelines, and a transcribed podcast. This makes it useful for comparing retrieval strategies without hiding the behavior behind a very large index.

## Core vs Advanced Features

| Feature | Level |
|---------|-------|
| Chunking with metadata | Core |
| OpenAI embeddings + Pinecone | Core |
| Baseline retrieval | Core |
| Metadata filtering | Core |
| Query enhancement | Core |
| Evaluation: Precision@5, MRR, NDCG@5 | Core |
| LLM-based relevance scoring | **Advanced** |
| Cohere Rerank | **Advanced** |
| CrossEncoder reranking | **Advanced / Optional** |


---
## Section 0 — Setup and Environment

This section loads dependencies, reads API keys from `.env`, and defines shared constants such as embedding model, index name, and retrieval cutoffs. The notebook treats external services as optional at setup time so missing keys produce clear messages instead of unclear import or client errors.


In [1]:
import os
import json
import time
import warnings
import importlib.util
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from pinecone import Pinecone, ServerlessSpec
except ImportError:
    Pinecone = None
    ServerlessSpec = None

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path=Path('.env'), override=False)

# Constants
OPENAI_MODEL   = 'gpt-4o-mini'
EMBED_MODEL    = 'text-embedding-3-small'
EMBED_DIM      = 1536
PINECONE_INDEX = os.getenv('PINECONE_INDEX_NAME', 'relevance-scoring-lab')
INITIAL_TOP_K  = 12
FINAL_TOP_K    = 5
FORCE_REINDEX  = False  # set True to re-embed and re-upload all chunks

OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY')

# Clients are optional so the notebook can still run and explain missing setup.
oai = OpenAI(api_key=OPENAI_API_KEY) if OpenAI and OPENAI_API_KEY else None
pc  = Pinecone(api_key=PINECONE_API_KEY) if Pinecone and PINECONE_API_KEY else None
index = None

HAS_OPENAI   = oai is not None
HAS_PINECONE = pc is not None
HAS_COHERE   = bool(COHERE_API_KEY) and importlib.util.find_spec('cohere') is not None


def pinecone_eq(field: str, value) -> Dict:
    return {field: {'$eq': value}}


print('Environment loaded')
print(f'  OpenAI model  : {OPENAI_MODEL}')
print(f'  Embed model   : {EMBED_MODEL} ({EMBED_DIM} dimensions)')
print(f'  Pinecone index: {PINECONE_INDEX}')
print(f'  Initial top-k : {INITIAL_TOP_K}  |  Final top-k: {FINAL_TOP_K}')
print('\nAPI/package status:')
print(f'  OpenAI   : {"ready" if HAS_OPENAI else "missing package or OPENAI_API_KEY"}')
print(f'  Pinecone : {"ready" if HAS_PINECONE else "missing package or PINECONE_API_KEY"}')
print(f'  Cohere   : {"ready" if HAS_COHERE else "missing package or COHERE_API_KEY"}')


Environment loaded
  OpenAI model  : gpt-4o-mini
  Embed model   : text-embedding-3-small (1536 dimensions)
  Pinecone index: relevance-scoring-lab
  Initial top-k : 12  |  Final top-k: 5

API/package status:
  OpenAI   : ready
  Pinecone : ready
  Cohere   : ready


---
## Section 1 — Data Loading and Inspection

`DOC_REGISTRY` is the source inventory for the lab. It explicitly lists the three professor-provided PDFs and the podcast transcript generated from the professor-provided audio file.

Each entry includes semantic retrieval metadata (`source_type`, `category`, `language`) and physical file metadata (`file_type`). The legacy `type` field is kept for the loader branch so the rest of the notebook remains backward compatible.

The inspection step confirms which files are available before chunking, so missing inputs are visible early and do not break the rest of the notebook.


In [43]:
DOC_REGISTRY = [
    {
        # EU AI Act legal text used for regulation-focused retrieval and evaluation queries.
        "path": "data/pdfs/eu_ai_act.pdf",
        "doc_id": "eu_ai_act",
        "doc_title": "EU AI Act",
        "category": "regulation",
        "source_type": "legal_document",
        "type": "pdf",
        "file_type": "pdf",
        "language": "en",
    },
    {
        # Professor-provided HLEG Ethics Guidelines for Trustworthy AI, English version.
        "path": "data/pdfs/ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419.pdf",
        "doc_id": "hleg_guidelines_en",
        "doc_title": "HLEG Ethics Guidelines for Trustworthy AI",
        "category": "class_material",
        "source_type": "legal_document",
        "type": "pdf",
        "file_type": "pdf",
        "language": "en",
    },
    {
        # Professor-provided HLEG Ethics Guidelines for Trustworthy AI, French version.
        "path": "data/pdfs/ethics_guidelines_for_trustworthy_ai-fr_87FE7A3C-D03D-9305-81A653DDA84B5A60_60427.pdf",
        "doc_id": "hleg_guidelines_fr",
        "doc_title": "HLEG Ethics Guidelines for Trustworthy AI",
        "category": "class_material",
        "source_type": "legal_document",
        "type": "pdf",
        "file_type": "pdf",
        "language": "fr",
    },
    {
        # Transcript generated from the professor-provided Trustworthy AI podcast audio file.
        "path": "data/transcripts/trustworthy_ai_podcast.txt",
        "doc_id": "trustworthy_ai_podcast",
        "doc_title": "The Blueprint for Trustworthy AI Podcast",
        "category": "podcast",
        "source_type": "podcast_transcript",
        "type": "transcript",
        "file_type": "transcript",
        "language": "en",
    },
]


In [44]:
from pypdf import PdfReader

print('File inventory:')
for doc in DOC_REGISTRY:
    p = Path(doc['path'])
    if not p.exists():
        print(f'  MISSING  {doc["path"]}')
        continue
    if doc['type'] == 'pdf':
        reader = PdfReader(doc['path'])
        print(f'  OK  {doc["doc_id"]:30s}  {len(reader.pages):4d} pages  [{doc["category"]}]')
    else:
        chars = len(p.read_text(encoding='utf-8'))
        print(f'  OK  {doc["doc_id"]:30s}  {chars:7,} chars  [{doc["category"]}]')

File inventory:
  OK  eu_ai_act                        144 pages  [regulation]
  OK  hleg_guidelines_en                41 pages  [class_material]
  OK  hleg_guidelines_fr                56 pages  [class_material]
  MISSING  data/transcripts/trustworthy_ai_podcast.txt


### 1.3 — Audio Transcription (optional)

The audio has already been transcribed and saved to `data/transcripts/trustworthy_ai_podcast.txt`. This cell remains in the notebook for reproducibility: if the transcript is missing, it compresses oversized audio when needed and uses OpenAI transcription to recreate the text file.

For the final run, the important result is that the podcast transcript exists and is included in the indexed corpus.


In [ ]:
import shutil
import subprocess

print('Audio transcription cell: compressed-upload version')

AUDIO_PATH = Path('data/Audio_Podcast/The_Blueprint_For_Trustworthy_AI.m4a')
TRANSCRIPT_OUT = Path('data/transcripts/trustworthy_ai_podcast.txt')
MAX_AUDIO_UPLOAD_BYTES = 25 * 1024 * 1024

if not AUDIO_PATH.exists():
    audio_candidates = sorted(Path('data/Audio_Podcast').glob('*.m4a'))
    if audio_candidates:
        AUDIO_PATH = audio_candidates[0]
        print(f'Using audio file: {AUDIO_PATH}')


def prepare_transcription_audio(audio_path: Path) -> Optional[Path]:
    if not audio_path.exists():
        return None
    if audio_path.stat().st_size <= MAX_AUDIO_UPLOAD_BYTES:
        return audio_path

    compressed_path = audio_path.with_name(f'{audio_path.stem}_compressed_for_transcription.mp3')
    if compressed_path.exists() and compressed_path.stat().st_size <= MAX_AUDIO_UPLOAD_BYTES:
        print(f'Using existing compressed audio: {compressed_path}')
        return compressed_path

    ffmpeg = shutil.which('ffmpeg')
    if not ffmpeg:
        size_mb = audio_path.stat().st_size / (1024 * 1024)
        print(f'Audio is {size_mb:.1f} MB, which exceeds the 25 MB transcription upload limit.')
        print('Install ffmpeg or provide a smaller audio file, then rerun this cell.')
        return None

    print(f'Audio is {audio_path.stat().st_size / (1024 * 1024):.1f} MB; compressing for transcription ...')
    cmd = [
        ffmpeg, '-y', '-i', str(audio_path),
        '-vn', '-ac', '1', '-ar', '16000', '-b:a', '32k',
        str(compressed_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if compressed_path.stat().st_size > MAX_AUDIO_UPLOAD_BYTES:
        size_mb = compressed_path.stat().st_size / (1024 * 1024)
        print(f'Compressed audio is still too large ({size_mb:.1f} MB); split the file before transcription.')
        return None

    print(f'Compressed audio saved: {compressed_path} ({compressed_path.stat().st_size / (1024 * 1024):.1f} MB)')
    return compressed_path


UPLOAD_AUDIO_PATH = prepare_transcription_audio(AUDIO_PATH)

if UPLOAD_AUDIO_PATH and not TRANSCRIPT_OUT.exists():
    if not HAS_OPENAI:
        print(f'Audio exists at {UPLOAD_AUDIO_PATH}, but OpenAI is not configured; skipping transcription.')
    else:
        print(f'Transcribing {UPLOAD_AUDIO_PATH} — this may take a minute ...')
        with UPLOAD_AUDIO_PATH.open('rb') as f:
            transcript = oai.audio.transcriptions.create(model='whisper-1', file=f)
        TRANSCRIPT_OUT.parent.mkdir(parents=True, exist_ok=True)
        TRANSCRIPT_OUT.write_text(transcript.text, encoding='utf-8')
        print(f'Saved transcript to {TRANSCRIPT_OUT}')
elif TRANSCRIPT_OUT.exists():
    print(f'Transcript already exists: {TRANSCRIPT_OUT}')
    print(f'  Size: {len(TRANSCRIPT_OUT.read_text()):,} chars')
else:
    print(f'Audio file not found at {AUDIO_PATH}; transcript document will be skipped.')



---
## Section 2 — Chunking and Metadata Tagging

This section converts each document into retrieval-sized chunks and attaches flat metadata from `DOC_REGISTRY`. Good metadata matters because it lets the retriever constrain search by semantic source type, source category, document ID, file type, or language before vector ranking happens.

**Metadata schema for legal documents:**
```python
{
    'source_type': 'legal_document',
    'file_type'  : 'pdf',
    'doc_id'     : 'eu_ai_act',
    'doc_title'  : 'EU AI Act',
    'category'   : 'regulation',
    'page'       : 12,
    'section'    : '',
    'chunk_index': 3,
    'chunk_id'   : 'eu_ai_act_p12_c3',
    'parent_id'  : 'eu_ai_act_p12',
    'char_count' : 412,
    'language'   : 'en',
}
```

Podcast chunks use `source_type='podcast_transcript'`, `file_type='transcript'`, and `category='podcast'`. The sample output below is a sanity check that chunk IDs, text, and metadata are aligned before indexing.

**Pinecone-filterable fields:** `source_type`, `file_type`, `category`, `doc_id`, `language`


In [5]:
from helpers.chunker import chunk_pdf, chunk_transcript

all_chunks: List[Dict] = []

for doc in DOC_REGISTRY:
    p = Path(doc['path'])
    if not p.exists():
        print(f'Skipping {doc["doc_id"]} — file not found')
        continue
    if doc['type'] == 'pdf':
        chunks = chunk_pdf(
            path=doc['path'],
            doc_id=doc['doc_id'],
            doc_title=doc['doc_title'],
            category=doc['category'],
            source_type=doc['source_type'],
            file_type=doc['file_type'],
            language=doc['language'],
        )
    else:
        chunks = chunk_transcript(
            path=doc['path'],
            doc_id=doc['doc_id'],
            doc_title=doc['doc_title'],
            source_type=doc['source_type'],
            file_type=doc['file_type'],
            language=doc['language'],
        )
    all_chunks.extend(chunks)
    print(f'  {doc["doc_id"]:30s}  {len(chunks):4d} chunks')

print(f'\nTotal chunks: {len(all_chunks)}')


  eu_ai_act                              405 chunks
  hleg_guidelines_en                     104 chunks
  hleg_guidelines_fr                     136 chunks
  trustworthy_ai_podcast                  10 chunks

Total chunks: 655


In [6]:
import random
random.seed(42)

for c in random.sample(all_chunks, min(3, len(all_chunks))):
    print('=' * 60)
    print(f'chunk_id : {c["chunk_id"]}')
    meta_display = {k: v for k, v in c['metadata'].items()}
    print(f'metadata : {json.dumps(meta_display, indent=2)}')
    print(f'text     : {c["text"][:200]}...')

chunk_id : eu_ai_act_p31_c2
metadata : {
  "source_type": "legal_document",
  "doc_id": "eu_ai_act",
  "doc_title": "EU AI Act",
  "category": "regulation",
  "page": 31,
  "section": "",
  "chunk_index": 114,
  "chunk_id": "eu_ai_act_p31_c2",
  "parent_id": "eu_ai_act_p31",
  "char_count": 1997,
  "language": "en"
}
text     : Regulation, in line with the state of the ar t, to promote inno vation as well as comp etitiveness and gro wth in the 
sing le market. Comp liance with har monised standards as defined in Ar ticle 2, ...
chunk_id : eu_ai_act_p8_c0
metadata : {
  "source_type": "legal_document",
  "doc_id": "eu_ai_act",
  "doc_title": "EU AI Act",
  "category": "regulation",
  "page": 8,
  "section": "",
  "chunk_index": 25,
  "chunk_id": "eu_ai_act_p8_c0",
  "parent_id": "eu_ai_act_p8",
  "char_count": 1892,
  "language": "en"
}
text     : (27) While the r isk -based approac h is the basis f or a propor tionate and effe ctive set of binding r ules, it is imp or tant to 
recall t

---
## Section 3 — Embeddings and Pinecone Indexing

Chunks are embedded with `text-embedding-3-small`, which produces 1536-dimensional vectors. The Pinecone index is checked against that dimension before upserting, because a dimension mismatch would make retrieval fail.

The chunk text is stored in metadata as `text` so retrieval returns both the similarity score and the source content needed for reranking, evaluation, and RAG answer generation.


In [7]:
from helpers.embedder import DIMENSION

if DIMENSION != EMBED_DIM:
    raise ValueError(f'Embedding dimension mismatch: notebook={EMBED_DIM}, helper={DIMENSION}')

if not HAS_PINECONE:
    print('Skipping Pinecone index setup because Pinecone is not configured.')
    index = None
else:
    listed = pc.list_indexes()
    existing = listed.names() if hasattr(listed, 'names') else [getattr(idx, 'name', idx.get('name')) for idx in listed]
    if PINECONE_INDEX not in existing:
        print(f'Creating index "{PINECONE_INDEX}" ...')
        pc.create_index(
            name=PINECONE_INDEX,
            dimension=DIMENSION,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1'),
        )
        print('Index created.')
    else:
        print(f'Index "{PINECONE_INDEX}" already exists.')

    index_desc = pc.describe_index(PINECONE_INDEX)
    index_dim = getattr(index_desc, 'dimension', None)
    if index_dim is None and isinstance(index_desc, dict):
        index_dim = index_desc.get('dimension')
    if index_dim != DIMENSION:
        raise ValueError(f'Pinecone index dimension is {index_dim}, expected {DIMENSION}. Create a matching index or change PINECONE_INDEX_NAME.')

    index = pc.Index(PINECONE_INDEX)
    print(index.describe_index_stats())


Creating index "relevance-scoring-lab" ...
Index created.
DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


In [8]:
from helpers.embedder import embed_texts

BATCH_SIZE = 100

def upsert_all_chunks(chunks: List[Dict], force: bool = False) -> None:
    if index is None:
        print('Skipping upsert because Pinecone is not configured.')
        return
    if not HAS_OPENAI:
        print('Skipping upsert because OpenAI embeddings are not configured.')
        return
    if not chunks:
        print('No chunks available to upsert. Add source PDFs/transcripts under data/ first.')
        return

    stats = index.describe_index_stats()
    if stats.total_vector_count > 0 and not force:
        print(f'Index already has {stats.total_vector_count} vectors.')
        print('Set FORCE_REINDEX = True to re-upload.')
        return

    print(f'Embedding and upserting {len(chunks)} chunks in batches of {BATCH_SIZE} ...')
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch = chunks[i:i + BATCH_SIZE]
        texts = [c['text'] for c in batch]
        embeddings = embed_texts(texts)
        vectors = [
            (c['chunk_id'], emb, {**c['metadata'], 'text': c['text']})
            for c, emb in zip(batch, embeddings)
        ]
        index.upsert(vectors=vectors)
    print(f'Done. Upserted {len(chunks)} vectors.')

upsert_all_chunks(all_chunks, force=FORCE_REINDEX)


Embedding and upserting 655 chunks in batches of 100 ...
Done. Upserted 655 vectors.


In [9]:
if index is None:
    print('Skipping index stats because Pinecone is not configured.')
else:
    stats = index.describe_index_stats()
    index_desc = pc.describe_index(PINECONE_INDEX)
    index_dim = getattr(index_desc, 'dimension', None)
    if index_dim is None and isinstance(index_desc, dict):
        index_dim = index_desc.get('dimension')
    print(f'Total vectors : {stats.total_vector_count}')
    print(f'Dimension     : {index_dim}')


Total vectors : 655
Dimension     : 1536


---
## Section 4 — Baseline Retrieval

The baseline pipeline embeds the query and retrieves the nearest chunks by cosine similarity. It uses no reranking, no metadata filter, and no query enhancement.

This baseline is the control condition for the lab. In the final labeled evaluation, it performed best overall, which means the embedding model and corpus were already well matched for the test queries.


In [10]:
from helpers.embedder import embed_query
from helpers.retriever import retrieve

TEST_QUERIES = [
    'What AI practices are prohibited under the EU AI Act?',
    'What are the key principles of trustworthy AI?',
    'How does the EU AI Act define high-risk AI systems?',
]

def show_results(results: List[Dict], title: str = 'Results') -> None:
    print(f'\n{"=" * 65}')
    print(f'  {title}')
    print(f'{"=" * 65}')
    for i, r in enumerate(results, 1):
        doc  = r['metadata'].get('doc_id', '?')
        page = r['metadata'].get('page', '-')
        print(f'  [{i}] score={r["score"]:.4f}  doc={doc}  page={page}')
        print(f'       {r["text"][:180]}...')

In [11]:
query = TEST_QUERIES[0]
print(f'Query: {query}')

baseline_results = []
baseline_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping baseline retrieval because OpenAI and Pinecone are both required.')
else:
    t0 = time.time()
    query_emb        = embed_query(query)
    baseline_results = retrieve(query_emb, top_k=INITIAL_TOP_K)
    baseline_latency = time.time() - t0

    show_results(baseline_results[:FINAL_TOP_K], f'Baseline — top {FINAL_TOP_K}')
    print(f'\nLatency: {baseline_latency:.2f}s')


Query: What AI practices are prohibited under the EU AI Act?

  Baseline — top 5
  [1] score=0.6470  doc=eu_ai_act  page=9
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the deplo y er , namely...
  [2] score=0.6647  doc=eu_ai_act  page=12
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because that practice adds to t...
  [3] score=0.6059  doc=hleg_guidelines_en  page=8
       sources include, but are not limit ed to : EU p rimary law (the Treaties of the  European Union and its Charter of 
Fundamental Rights ), EU s econdary law (such as the General Dat...
  [4] score=0.6266  doc=eu_ai_act  page=45
       (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and oblig ations f or operators of suc h systems;
(d) har monise

In [12]:
# Quick smoke test: confirm baseline retrieval returns readable evidence for several query types.
if index is None or not HAS_OPENAI:
    print('Skipping baseline retrieval loop because OpenAI and Pinecone are both required.')
else:
    for q in TEST_QUERIES:
        qe  = embed_query(q)
        res = retrieve(qe, top_k=FINAL_TOP_K)
        show_results(res, q[:60])



  What AI practices are prohibited under the EU AI Act?
  [1] score=0.6710  doc=eu_ai_act  page=51
       Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite racy of 
their staff and other...
  [2] score=0.6647  doc=eu_ai_act  page=12
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because that practice adds to t...
  [3] score=0.6470  doc=eu_ai_act  page=9
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the deplo y er , namely...
  [4] score=0.6266  doc=eu_ai_act  page=45
       (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and oblig ations f or operators of suc h systems;
(d) har monised transparency r ule...
  [5] sc

---
## Section 5 — Metadata Filtering

Metadata filtering narrows the candidate set before vector similarity ranking. This is useful when the query should target a known source type, such as legal documents for EU AI Act obligations or podcast transcript chunks for discussion-style trustworthy AI content.

The final run confirms filtering works for both `legal_document` and `podcast_transcript` sources. Filtering is not a reranker; it improves retrieval by removing irrelevant parts of the corpus before ranking.

**Filterable fields:** `source_type`, `category`, `doc_id`, `language`


In [13]:
query = TEST_QUERIES[0]

if index is None or not HAS_OPENAI:
    reg_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    reg_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('category', 'regulation'))
    show_results(reg_results[:FINAL_TOP_K], 'Filtered: category=regulation')



  Filtered: category=regulation
  [1] score=0.6470  doc=eu_ai_act  page=9
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the deplo y er , namely...
  [2] score=0.6647  doc=eu_ai_act  page=12
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because that practice adds to t...
  [3] score=0.6046  doc=eu_ai_act  page=41
       requesting measures from provid ers of g eneral-pur pose AI models. When conducting evaluations, in order to make 
use of independent exper tise, the AI Off ice should be able to i...
  [4] score=0.6710  doc=eu_ai_act  page=51
       Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite racy of 
their staff and other...
  [5] score=0.6266  doc=eu_ai_ac

In [14]:
query = TEST_QUERIES[1]

if index is None or not HAS_OPENAI:
    pod_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    pod_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('source_type', 'podcast_transcript'))
    show_results(pod_results[:FINAL_TOP_K], 'Filtered: source_type=podcast_transcript')



  Filtered: source_type=podcast_transcript
  [1] score=0.6407  doc=trustworthy_ai_podcast  page=-
       So, imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over ha...
  [2] score=0.6311  doc=trustworthy_ai_podcast  page=-
       out how to operationalize that. We aren't just talking philosophy here. We're looking at how you take an abstract conce...
  [3] score=0.6296  doc=trustworthy_ai_podcast  page=-
       humans on their behavior to deny them services or rights endangers human autonomy. It's fundamentally anti-democratic. ...
  [4] score=0.5806  doc=trustworthy_ai_podcast  page=-
       specific. Fundamental rights. Yes. The EU Charter, specifically. This is so crucial because it changes the framing. AI...
  [5] score=0.5630  doc=trustworthy_ai_podcast  page=-
       environmental well-being. This is the first time I've really seen green AI emphasized in a foundational document like ...


In [15]:
query = TEST_QUERIES[2]

if index is None or not HAS_OPENAI:
    doc_results = []
    print('Skipping filtered retrieval because OpenAI and Pinecone are both required.')
else:
    qe = embed_query(query)
    doc_results = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=pinecone_eq('doc_id', 'eu_ai_act'))
    show_results(doc_results[:FINAL_TOP_K], 'Filtered: doc_id=eu_ai_act')

    # Show how filtering changes the source mix in the top results.
    unfiltered  = retrieve(qe, top_k=FINAL_TOP_K)
    docs_in_un  = [r['metadata'].get('doc_id') for r in unfiltered]
    docs_in_fil = [r['metadata'].get('doc_id') for r in doc_results[:FINAL_TOP_K]]
    print(f'\nUnfiltered doc_ids : {docs_in_un}')
    print(f'Filtered doc_ids   : {docs_in_fil}')



  Filtered: doc_id=eu_ai_act
  [1] score=0.7218  doc=eu_ai_act  page=127
       ANNEX III
High-r isk AI sys tems referred to in Ar ticle 6(2)
High-r isk AI systems pursuant to Ar ticle 6(2) are the AI systems listed in an y of the f ollo wing areas:
1. Biometr...
  [2] score=0.7058  doc=eu_ai_act  page=14
       European P arliament and of the Council (
30
), and Regulation (EU) 2019/2144 of the European Parliament and of the 
Council (
31
), it is appropr iate to amend those acts t o ensu...
  [3] score=0.7193  doc=eu_ai_act  page=55
       kno wledge, economic or social circumstances, or age;
(i) the exte nt to which the outcome produced inv olving an AI system is easily cor r igible or reversible, taking into accoun...
  [4] score=0.7036  doc=eu_ai_act  page=14
       par ticular , the case f or Regulations (EU) 2017/745 and (EU) 2017/746, where a third-par ty conf or mity assessment is 
pro vided f or medium-r isk and high-r isk products.
(52) ...
  [5] score=0.6927  doc=eu_ai_act

---
## Section 6 — Query Enhancement

Query enhancement changes the search text before embedding it. These methods can help when user wording differs from document wording, especially in legal and compliance corpora where precise terms matter.

| Strategy | Purpose |
|----------|---------|
| **Query rewriting** | Rephrase the query using terms closer to the source documents |
| **Sub-query decomposition** | Split compound questions into smaller retrieval targets |
| **Step-back prompting** | Retrieve broader context around the original question |
| **Query expansion** | Add related terms and alternative phrasings |

The retrieved chunks are compared against the original query results to check whether enhancement changes the evidence set.


In [17]:
def rewrite_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Rewrite the following search query to improve retrieval from a legal and AI policy corpus.\n'
        'Make it more precise and use relevant legal/technical terminology.\n\n'
        f'Original query: {query}\n\n'
        'Rewritten query (one sentence, no explanation):'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=100,
    )
    return resp.choices[0].message.content.strip()


def decompose_query(query: str) -> List[str]:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query as the only sub-query.')
        return [query]
    prompt = (
        'Break the following query into 2-3 simpler sub-queries that together cover the full question.\n'
        'Return one sub-query per line with no numbering or bullet points.\n\n'
        f'Query: {query}'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=150,
    )
    return [ln.strip() for ln in resp.choices[0].message.content.strip().split('\n') if ln.strip()]


def step_back_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Given the specific question below, write a broader, more general question that would '
        'help retrieve useful background context.\n\n'
        f'Specific question: {query}\n\n'
        'General step-back question (one sentence):'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=80,
    )
    return resp.choices[0].message.content.strip()


def expand_query(query: str) -> str:
    if not HAS_OPENAI:
        print('OpenAI is not configured; using the original query.')
        return query
    prompt = (
        'Expand the query by adding synonyms, related legal terms, and alternative phrasings. '
        'Return a single expanded query string.\n\n'
        f'Query: {query}\n\n'
        'Expanded query:'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=120,
    )
    return resp.choices[0].message.content.strip()


In [18]:
original   = TEST_QUERIES[0]
rewritten  = rewrite_query(original)
stepback   = step_back_query(original)
expanded   = expand_query(original)
sub_qs     = decompose_query(original)

print(f'Original   : {original}')
print(f'Rewritten  : {rewritten}')
print(f'Step-back  : {stepback}')
print(f'Expanded   : {expanded}')
print(f'Sub-queries:')
for sq in sub_qs:
    print(f'  {sq}')

Original   : What AI practices are prohibited under the EU AI Act?
Rewritten  : What artificial intelligence practices are explicitly prohibited by the EU Artificial Intelligence Act?
Step-back  : What are the key regulations and guidelines governing the use of artificial intelligence in the European Union?
Expanded   : What artificial intelligence practices, methodologies, or techniques are forbidden, restricted, or disallowed under the European Union Artificial Intelligence Act, including any unlawful, impermissible, or non-compliant uses of AI technologies, as well as associated regulations, guidelines, or legal frameworks governing AI deployment in the EU?
Sub-queries:
  What are the key provisions of the EU AI Act?
  What specific AI practices are classified as high-risk under the EU AI Act?
  Which AI practices are explicitly prohibited by the EU AI Act?


In [19]:
# Compare whether query rewriting changes the retrieved evidence set.
if index is None or not HAS_OPENAI:
    orig_res = []
    rew_res = []
    print('Skipping retrieval comparison because OpenAI and Pinecone are both required.')
else:
    print('--- Original query ---')
    orig_res = retrieve(embed_query(original), top_k=FINAL_TOP_K)
    for r in orig_res:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')

    print('\n--- Rewritten query ---')
    rew_res = retrieve(embed_query(rewritten), top_k=FINAL_TOP_K)
    for r in rew_res:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')

    # Overlap analysis
    orig_ids = {r['chunk_id'] for r in orig_res}
    rew_ids  = {r['chunk_id'] for r in rew_res}
    print(f'\nChunk overlap: {len(orig_ids & rew_ids)} / {FINAL_TOP_K}')


--- Original query ---
  [0.6710] eu_ai_act | Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to t...
  [0.6647] eu_ai_act | expand f acial recognition databases through the untarg eted scraping of f acial images from the int...
  [0.6470] eu_ai_act | inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI sy...
  [0.6266] eu_ai_act | (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and ...
  [0.6159] hleg_guidelines_en | trustworthy AI ( ethical and robust AI). These statements are hence not meant to provide legal advic...

--- Rewritten query ---
  [0.6643] eu_ai_act | Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to t...
  [0.6499] eu_ai_act | expand f acial recognition databases through the untarg eted scraping of f acial images from the int...
  [0.6344] eu_ai_act | inte ntion to distor t behavio 

In [20]:
# Retrieve each sub-query independently, then merge unique chunks for review.
seen_ids: set = set()
merged: List[Dict] = []

if index is None or not HAS_OPENAI:
    print('Skipping sub-query retrieval because OpenAI and Pinecone are both required.')
else:
    for sq in sub_qs:
        sq_results = retrieve(embed_query(sq), top_k=FINAL_TOP_K)
        for r in sq_results:
            if r['chunk_id'] not in seen_ids:
                merged.append(r)
                seen_ids.add(r['chunk_id'])

    print(f'Sub-query decomposition retrieved {len(merged)} unique chunks across {len(sub_qs)} sub-queries:')
    for r in merged[:FINAL_TOP_K]:
        print(f'  [{r["score"]:.4f}] {r["metadata"].get("doc_id")} | {r["text"][:100]}...')


Sub-query decomposition retrieved 15 unique chunks across 3 sub-queries:
  [0.6757] hleg_guidelines_en | sources include, but are not limit ed to : EU p rimary law (the Treaties of the  European Union and ...
  [0.6713] eu_ai_act | REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying ...
  [0.6657] eu_ai_act | guarante eing the unif or m protect ion of ove r r iding reasons of public inte rest and of r ights ...
  [0.6605] eu_ai_act | AI-based goods and ser vices, thus preventing Member Stat es from imp osing restr ictions on the dev...
  [0.6597] eu_ai_act | with the use of AI in cer tain wa ys, the prohibitions as well as the g eneral provis ions of this R...


---
## Section 7 — LLM-Based Relevance Scoring *(Advanced)*

This advanced reranking method asks `gpt-4o-mini` to score each retrieved candidate from 0 to 10 for relevance to the original query. The candidates are then sorted by the LLM score instead of the vector similarity score.

This approach is flexible because the scoring prompt can encode task-specific relevance criteria. The trade-off is latency and cost: each query requires one LLM call per candidate. In the final evaluation, LLM scoring did not outperform the baseline on this small labeled set.


In [21]:
from helpers.reranker import llm_score_relevance

query      = TEST_QUERIES[0]
candidates = []
llm_results = []
llm_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping LLM scoring because OpenAI and Pinecone are both required.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Scoring {len(candidates)} candidates with LLM ...')
    t0          = time.time()
    llm_results = llm_score_relevance(query, candidates, oai, top_n=FINAL_TOP_K)
    llm_latency = time.time() - t0

    print(f'\nLLM-Scored top {FINAL_TOP_K}  (latency: {llm_latency:.2f}s):')
    for i, r in enumerate(llm_results, 1):
        print(f'  [{i}] LLM={r["llm_score"]:.1f}  cosine={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


Scoring 12 candidates with LLM ...

LLM-Scored top 5  (latency: 9.13s):
  [1] LLM=5.0  cosine=0.6266  doc=eu_ai_act
       (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and oblig ations f or operators of suc h systems;
(d) har monise...
  [2] LLM=4.0  cosine=0.6647  doc=eu_ai_act
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because tha...
  [3] LLM=3.0  cosine=0.6710  doc=eu_ai_act
       Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite racy of 
t...
  [4] LLM=3.0  cosine=0.6470  doc=eu_ai_act
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the...
  [5] LLM=3.0  cosine=0.6046  doc=eu_ai_act
       requesting measur

In [22]:
# Compare whether LLM scoring changed the top-five ordering.
baseline_ids = [r['chunk_id'] for r in candidates[:FINAL_TOP_K]]
llm_ids      = [r['chunk_id'] for r in llm_results]

if not baseline_ids:
    print('Skipping rank-change summary because no candidates are available.')
else:
    print('Rank changes after LLM scoring:')
    print(f'  Baseline  : {baseline_ids}')
    print(f'  LLM-scored: {llm_ids}')
    moved = sum(1 for a, b in zip(baseline_ids, llm_ids) if a != b)
    print(f'  {moved} / {FINAL_TOP_K} positions changed')


Rank changes after LLM scoring:
  Baseline  : ['eu_ai_act_p51_c0', 'eu_ai_act_p12_c1', 'eu_ai_act_p9_c0', 'eu_ai_act_p45_c0', 'hleg_guidelines_en_p4_c2']
  LLM-scored: ['eu_ai_act_p45_c0', 'eu_ai_act_p12_c1', 'eu_ai_act_p51_c0', 'eu_ai_act_p9_c0', 'eu_ai_act_p41_c2']
  4 / 5 positions changed


---
## Section 8 — Cohere Reranking *(Advanced)*

Cohere Rerank scores each `(query, document)` pair after the initial vector search. The pattern is broad retrieval (`top_k=12`), rerank those candidates, then keep the top 5.

This is a standard two-stage retrieval setup: vector search provides recall, and the reranker attempts to improve precision. In the final evaluation, Cohere matched the baseline on MRR but had lower Precision@5 and NDCG@5, so it did not improve this particular labeled set.


In [23]:
from helpers.reranker import cohere_rerank

query      = TEST_QUERIES[0]
candidates = []
cohere_results = []
cohere_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping Cohere reranking because OpenAI and Pinecone are both required for candidate retrieval.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Reranking {len(candidates)} candidates with Cohere ...')
    t0              = time.time()
    cohere_results  = cohere_rerank(query, candidates, top_n=FINAL_TOP_K)
    cohere_latency  = time.time() - t0

    print(f'\nCohere Reranked top {FINAL_TOP_K}  (latency: {cohere_latency:.2f}s):')
    for i, r in enumerate(cohere_results, 1):
        rerank_score = r.get('rerank_score', r.get('score', 0.0))
        print(f'  [{i}] rerank={rerank_score:.4f}  cosine={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


Reranking 12 candidates with Cohere ...

Cohere Reranked top 5  (latency: 4.09s):
  [1] rerank=0.8452  cosine=0.6710  doc=eu_ai_act
       Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite racy of 
t...
  [2] rerank=0.8371  cosine=0.6470  doc=eu_ai_act
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the...
  [3] rerank=0.7641  cosine=0.6647  doc=eu_ai_act
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because tha...
  [4] rerank=0.6036  cosine=0.6059  doc=hleg_guidelines_en
       sources include, but are not limit ed to : EU p rimary law (the Treaties of the  European Union and its Charter of 
Fundamental Rights ), EU s econdary law (suc...
  [5] rerank=0.5724  cosi

In [24]:
# Compare whether Cohere changed the top-five ordering.
baseline_ids = [r['chunk_id'] for r in candidates[:FINAL_TOP_K]]
cohere_ids   = [r['chunk_id'] for r in cohere_results]

if not baseline_ids:
    print('Skipping rank-change summary because no candidates are available.')
else:
    print('Rank changes after Cohere reranking:')
    print(f'  Baseline: {baseline_ids}')
    print(f'  Cohere  : {cohere_ids}')
    moved = sum(1 for a, b in zip(baseline_ids, cohere_ids) if a != b)
    print(f'  {moved} / {FINAL_TOP_K} positions changed')


Rank changes after Cohere reranking:
  Baseline: ['eu_ai_act_p51_c0', 'eu_ai_act_p12_c1', 'eu_ai_act_p9_c0', 'eu_ai_act_p45_c0', 'hleg_guidelines_en_p4_c2']
  Cohere  : ['eu_ai_act_p51_c0', 'eu_ai_act_p9_c0', 'eu_ai_act_p12_c1', 'hleg_guidelines_en_p8_c1', 'eu_ai_act_p45_c0']
  4 / 5 positions changed


---
## Section 9 — CrossEncoder Reranking *(Advanced — Optional)*

The optional local CrossEncoder scores `(query, document)` pairs without a hosted rerank API. It is useful for experimentation when local model downloads are acceptable.

This section is intentionally optional. If `sentence-transformers` or the model is unavailable, the helper returns the baseline order with a warning so the notebook can still run top to bottom.


In [25]:
from helpers.reranker import crossencoder_rerank

query      = TEST_QUERIES[0]
candidates = []
ce_results = []
ce_latency = 0.0

if index is None or not HAS_OPENAI:
    print('Skipping CrossEncoder reranking because OpenAI and Pinecone are both required for candidate retrieval.')
else:
    candidates = retrieve(embed_query(query), top_k=INITIAL_TOP_K)

    print(f'Reranking {len(candidates)} candidates with CrossEncoder ...')
    t0          = time.time()
    ce_results  = crossencoder_rerank(query, candidates, top_n=FINAL_TOP_K)
    ce_latency  = time.time() - t0

    print(f'\nCrossEncoder top {FINAL_TOP_K}  (latency: {ce_latency:.2f}s):')
    for i, r in enumerate(ce_results, 1):
        score = r.get('crossencoder_score', r['score'])
        print(f'  [{i}] ce_score={score:.4f}  doc={r["metadata"].get("doc_id")}')
        print(f'       {r["text"][:160]}...')


Reranking 12 candidates with CrossEncoder ...
sentence-transformers not installed.
To enable: uncomment sentence-transformers and torch in requirements.txt, then pip install.

CrossEncoder top 5  (latency: 0.00s):
  [1] ce_score=0.6710  doc=eu_ai_act
       Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a suffi cient level of AI lite racy of 
t...
  [2] ce_score=0.6647  doc=eu_ai_act
       expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f ootage, should be prohibite d because tha...
  [3] ce_score=0.6470  doc=eu_ai_act
       inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outside 
the control of the provid er or the...
  [4] ce_score=0.6266  doc=eu_ai_act
       (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and oblig ations f or 

---
## Section 10 — Full RAG Pipeline with Reranking

This section combines the earlier pieces into an end-to-end RAG function: optional query enhancement, vector retrieval, optional reranking, context assembly, and answer generation.

The `reranker` argument switches between baseline, LLM scoring, Cohere, and CrossEncoder variants. This makes the downstream answer generation comparable across retrieval strategies while keeping the prompt and corpus fixed.


In [27]:
def rag_pipeline(
    query: str,
    reranker: str = 'cohere',           # 'none' | 'llm' | 'cohere' | 'crossencoder'
    metadata_filter: Optional[Dict] = None,
    enhance_query: bool = True,
) -> str:
    if index is None or not HAS_OPENAI:
        return 'RAG pipeline skipped because OpenAI and Pinecone are both required.'

    # 1. Query enhancement
    retrieval_query = rewrite_query(query) if enhance_query else query

    # 2. Embed + retrieve broad candidate set
    qe         = embed_query(retrieval_query)
    candidates = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=metadata_filter)

    # 3. Rerank
    if reranker == 'cohere':
        top_chunks = cohere_rerank(query, candidates, top_n=FINAL_TOP_K)
    elif reranker == 'llm':
        top_chunks = llm_score_relevance(query, candidates, oai, top_n=FINAL_TOP_K)
    elif reranker == 'crossencoder':
        top_chunks = crossencoder_rerank(query, candidates, top_n=FINAL_TOP_K)
    else:
        top_chunks = candidates[:FINAL_TOP_K]

    # 4. Build context
    context = '\n\n---\n\n'.join(
        f'[{c["metadata"].get("doc_id")} | p.{c["metadata"].get("page", "-")}]\n{c["text"]}'
        for c in top_chunks
    )

    # 5. Generate
    system = (
        'You are an expert on EU AI policy and trustworthy AI. '
        'Answer the question using only the provided context. '
        'Cite the document and page when possible. '
        'If the context does not contain the answer, say so clearly.'
    )
    resp = oai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
        temperature=0.2,
        max_tokens=500,
    )
    return resp.choices[0].message.content


In [28]:
# Run one end-to-end example using Cohere reranking.
query  = 'What are the prohibited AI practices in the EU AI Act?'
answer = rag_pipeline(query, reranker='cohere', metadata_filter=pinecone_eq('category', 'regulation'))
print(f'Query: {query}\n')
print(answer)


Query: What are the prohibited AI practices in the EU AI Act?

The prohibited AI practices in the EU AI Act include:

1. The use of AI systems that deploy subliminal techniques or manipulative techniques to materially distort the behavior of individuals, impairing their ability to make informed decisions and causing significant harm (Article 5, p.51).
   
2. The use of AI systems that exploit vulnerabilities of individuals or specific groups due to age, disability, or social/economic situations, leading to significant harm (Article 5, p.51).

3. The use of AI systems for evaluating or classifying individuals based on their social behavior or inferred personal characteristics, leading to detrimental treatment in unrelated social contexts (Article 5, p.51).

4. The use of AI systems for risk assessments of individuals based solely on profiling or assessing personality traits, without objective and verifiable facts linking them to criminal activity (Article 5, p.51).

5. The creation or e

In [29]:
# Compare generated answers while changing only the retrieval/reranking path.
query = 'What transparency obligations apply to AI systems?'

print('=== BASELINE (no reranking) ===')
baseline_answer = rag_pipeline(query, reranker='none', enhance_query=False)
print(baseline_answer)

print('\n=== COHERE RERANKED ===')
cohere_answer = rag_pipeline(query, reranker='cohere')
print(cohere_answer)


=== BASELINE (no reranking) ===
The transparency obligations for AI systems, particularly those intended to interact directly with natural persons, include the following:

1. **Identification**: AI systems must be designed and developed in such a way that users are informed they are interacting with an AI system, unless this is obvious to a reasonably well-informed person (eu_ai_act, p.82).

2. **Information Provision**: High-risk AI systems must be accompanied by appropriate information in the form of instructions for use, detailing the characteristics, capabilities, and limitations of the AI system. This includes information on foreseeable circumstances related to its use, potential risks to health, safety, and fundamental rights, and relevant human oversight measures (eu_ai_act, p.21).

3. **Traceability and Documentation**: The data sets and processes that yield the AI system’s decisions should be documented to allow for traceability and increase transparency. This includes data ga

---
## Section 11 — Manual Evaluation

The evaluation uses manually labeled chunk IDs from `eval_queries.json`. This keeps the comparison focused on retrieval quality rather than answer style.

### Ground-truth process

1. Run baseline retrieval for each evaluation query.
2. Review the top candidate chunks manually.
3. Assign relevance labels:
   - `2` = highly relevant and directly useful
   - `1` = partially relevant or useful context
   - `0` = not relevant
4. Store relevant chunk IDs and graded labels in `eval_queries.json`.
5. Compare each retrieval method with Precision@5, MRR, and NDCG@5.

The final metrics should be read as a small labeled test set, not a universal benchmark. They are sufficient to compare behavior in this lab corpus.


In [31]:
from helpers.evaluator import load_eval_queries, evaluate_pipeline

eval_queries = load_eval_queries('eval_queries.json')

for eq in eval_queries:
    status = 'READY' if eq['relevant_chunk_ids'] else 'NEEDS LABELS'
    print(f'  [{status:12s}] {eq["query_id"]}: {eq["query"][:70]}')

labeled = sum(1 for eq in eval_queries if eq['relevant_chunk_ids'])
print(f'\n{labeled}/{len(eval_queries)} queries have labels.')
if labeled == 0:
    print('Run baseline retrieval below, review results, then update eval_queries.json.')

  [NEEDS LABELS] q1: What AI practices are prohibited under the EU AI Act?
  [NEEDS LABELS] q2: How does the EU AI Act define high-risk AI systems?
  [NEEDS LABELS] q3: What transparency obligations apply to AI systems under EU law?
  [NEEDS LABELS] q4: What are the key principles of trustworthy AI?
  [NEEDS LABELS] q5: How should AI systems handle fundamental rights in the EU?
  [NEEDS LABELS] q6: What conformity assessment procedures are required for high-risk AI?

0/6 queries have labels.
Run baseline retrieval below, review results, then update eval_queries.json.


In [32]:
# Print candidate chunks used to build or audit manual relevance labels.
print('Baseline results for manual labeling\n')
print('Copy relevant chunk_ids into eval_queries.json\n')
print('=' * 65)

if index is None or not HAS_OPENAI:
    print('Skipping manual-label retrieval because OpenAI and Pinecone are both required.')
else:
    for eq in eval_queries:
        f   = eq.get('filter') or None
        qe  = embed_query(eq['query'])
        res = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=f)
        print(f'\nQuery [{eq["query_id"]}]: {eq["query"]}')
        for i, r in enumerate(res[:8], 1):
            print(f'  [{i:2d}] id={r["chunk_id"]}  score={r["score"]:.4f}  doc={r["metadata"].get("doc_id")}')
            print(f'        {r["text"][:120]}...')


Baseline results for manual labeling

Copy relevant chunk_ids into eval_queries.json


Query [q1]: What AI practices are prohibited under the EU AI Act?
  [ 1] id=eu_ai_act_p51_c0  score=0.6710  doc=eu_ai_act
        Ar ticle 4
AI literacy
Provi ders and deplo y ers of AI syste ms shall take measures to ensure, to their best exte nt, a...
  [ 2] id=eu_ai_act_p12_c1  score=0.6647  doc=eu_ai_act
        expand f acial recognition databases through the untarg eted scraping of f acial images from the inter net or CCT V 
f o...
  [ 3] id=eu_ai_act_p9_c0  score=0.6470  doc=eu_ai_act
        inte ntion to distor t behavio ur where the distor tion results from f actors exter nal to the AI syste m which are outs...
  [ 4] id=eu_ai_act_p45_c0  score=0.6266  doc=eu_ai_act
        (b) prohibitions of cer tain AI practices;
(c) specific requirements f or high-r isk AI systems and oblig ations f or op...
  [ 5] id=eu_ai_act_p41_c2  score=0.6046  doc=eu_ai_act
        requesting measures from provid 

In [33]:
# Evaluate each retrieval strategy against the same labeled query set.
pipeline_results: Dict[str, List] = {
    'Baseline'      : [],
    'LLM Scoring'   : [],
    'Cohere Rerank' : [],
}

if index is None or not HAS_OPENAI:
    print('Skipping live evaluation because OpenAI and Pinecone are both required.')
    for _ in eval_queries:
        pipeline_results['Baseline'].append([])
        pipeline_results['LLM Scoring'].append([])
        pipeline_results['Cohere Rerank'].append([])
else:
    for eq in tqdm(eval_queries, desc='Evaluating queries'):
        q  = eq['query']
        f  = eq.get('filter') or None
        qe = embed_query(q)

        candidates = retrieve(qe, top_k=INITIAL_TOP_K, metadata_filter=f)

        pipeline_results['Baseline'].append(candidates[:FINAL_TOP_K])
        pipeline_results['LLM Scoring'].append(
            llm_score_relevance(q, candidates, oai, top_n=FINAL_TOP_K)
        )
        pipeline_results['Cohere Rerank'].append(
            cohere_rerank(q, candidates, top_n=FINAL_TOP_K)
        )

print('Done.')


Evaluating queries: 100%|██████████| 6/6 [01:04<00:00, 10.69s/it]

Done.


In [36]:
metrics = []
for name, results in pipeline_results.items():
    m = evaluate_pipeline(name, results, eval_queries, k=FINAL_TOP_K)
    metrics.append(m)

df_metrics = pd.DataFrame(metrics).set_index('pipeline')
print(df_metrics.to_string())

df_metrics.to_csv('outputs/eval_results.csv')
print('\nSaved to outputs/eval_results.csv')

               precision_at_5    mrr  ndcg_at_5
pipeline                                       
Baseline                0.933  0.917      0.895
LLM Scoring             0.633  0.917      0.671
Cohere Rerank           0.767  0.917      0.780

Saved to outputs/eval_results.csv


---
## Section 12 — Results and Conclusion

### Final Evaluation Results

| Pipeline | Precision@5 | MRR | NDCG@5 |
|----------|-------------|-----|--------|
| Baseline | 0.933 | 0.917 | 0.895 |
| Cohere Rerank | 0.767 | 0.917 | 0.780 |
| LLM Scoring | 0.633 | 0.917 | 0.671 |

### Interpretation

Baseline retrieval performed best on the labeled evaluation set. It had the highest Precision@5 and NDCG@5, while all three methods had the same MRR. This means the first relevant result was usually found at a similar rank, but the baseline kept more relevant chunks in the top five and preserved better graded relevance ordering.

Reranking did not outperform the baseline here because the corpus is relatively small, domain-focused, and well matched to the evaluation queries. The embedding baseline already retrieved strong candidate chunks, so the rerankers had limited room to improve and sometimes reordered useful evidence lower.

### Conclusion

For this lab submission, the best measured retrieval strategy is the baseline vector search with metadata filtering available when the user intent is source-specific. Reranking remains valuable in larger or noisier RAG systems, especially with heterogeneous corpora, ambiguous queries, weak first-stage retrieval, high-precision legal or compliance workflows, and larger candidate sets where the initial top results contain more near-misses.

The result is a useful negative finding: reranking is not automatically better. It should be evaluated against a strong baseline and adopted when the measured retrieval quality justifies the added cost, latency, and operational dependency.


In [37]:
print('=' * 65)
print('FINAL EVALUATION SUMMARY')
print('=' * 65)
print(df_metrics.to_string())

# Identify best-performing pipeline per metric
for metric in ['precision_at_5', 'mrr', 'ndcg_at_5']:
    if metric in df_metrics.columns:
        best = df_metrics[metric].idxmax()
        val  = df_metrics[metric].max()
        print(f'\nBest {metric:15s}: {best} ({val:.3f})')

FINAL EVALUATION SUMMARY
               precision_at_5    mrr  ndcg_at_5
pipeline                                       
Baseline                0.933  0.917      0.895
LLM Scoring             0.633  0.917      0.671
Cohere Rerank           0.767  0.917      0.780

Best precision_at_5: Baseline (0.933)
Best mrr           : Baseline (0.917)
Best ndcg_at_5     : Baseline (0.895)


In [38]:
# Measure one-query latency to show the cost of each reranking strategy.
if index is None or not HAS_OPENAI:
    print('Skipping latency comparison because OpenAI and Pinecone are both required.')
else:
    q  = TEST_QUERIES[0]
    qe = embed_query(q)
    candidates = retrieve(qe, top_k=INITIAL_TOP_K)

    timings = {}

    t0 = time.time(); candidates[:FINAL_TOP_K]; timings['baseline']     = time.time() - t0
    t0 = time.time(); cohere_rerank(q, candidates, top_n=FINAL_TOP_K);  timings['cohere']       = time.time() - t0
    t0 = time.time(); llm_score_relevance(q, candidates, oai, top_n=FINAL_TOP_K); timings['llm'] = time.time() - t0
    t0 = time.time(); crossencoder_rerank(q, candidates, top_n=FINAL_TOP_K);      timings['crossencoder'] = time.time() - t0

    df_latency = pd.DataFrame([{'pipeline': k, 'latency_s': round(v, 2)} for k, v in timings.items()])
    df_latency = df_latency.set_index('pipeline')
    print('\nLatency (single query):')
    print(df_latency.to_string())


sentence-transformers not installed.
To enable: uncomment sentence-transformers and torch in requirements.txt, then pip install.

Latency (single query):
              latency_s
pipeline               
baseline           0.00
cohere             0.45
llm                7.27
crossencoder       0.00
